In [1]:
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224

# Diviser les données en 80% train et 20% pour validation/test
train_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds) - val_size  # Reste pour test

val_ds = val_test_ds.take(val_size)  # Premier 50% pour validation
test_ds = val_test_ds.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds)}")
print(f"Nombre de batches dans val_ds: {len(val_ds)}")
print(f"Nombre de batches dans test_ds: {len(test_ds)}")

Found 11540 files belonging to 3 classes.
Using 9232 files for training.
Found 11540 files belonging to 3 classes.
Using 2308 files for validation.
Nombre de batches dans train_ds: 289
Nombre de batches dans val_ds: 36
Nombre de batches dans test_ds: 37


In [2]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds_maiis))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds_maiis) - val_size  # Reste pour test

val_ds_maiis = val_test_ds_maiis.take(val_size)  # Premier 50% pour validation
test_ds_maiis = val_test_ds_maiis.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_maiis)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_maiis)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_maiis)}")

Found 11480 files belonging to 3 classes.
Using 9184 files for training.
Found 11480 files belonging to 3 classes.
Using 2296 files for validation.
Nombre de batches dans train_ds: 287
Nombre de batches dans val_ds: 36
Nombre de batches dans test_ds: 36


In [1]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size_mixte = int(0.5 * len(val_test_ds_mixte))  # 50% de val_test_ds pour validation
test_size_mixte = len(val_test_ds_mixte) - val_size_mixte  # Reste pour test

val_ds_mixte = val_test_ds_maiis.take(val_size_mixte)  # Premier 50% pour validation
test_ds_mixte = val_test_ds_maiis.skip(val_size_mixte)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_mixte)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_mixte)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_mixte)}")

NameError: name 'tf' is not defined

In [4]:
# Afficher les classes
class_names = train_val_ds.class_names
print("Les classes sont :")
for i, class_name in enumerate(class_names):
    print(f"{i}: {class_name}")

Les classes sont :
0: legere
1: saine
2: severe


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet201  # Utiliser DenseNet201
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir les dimensions d'image
img_height, img_width = 224, 224

# Charger le modèle DenseNet201 pré-entraîné
base_model = DenseNet201(input_shape=(img_height, img_width, 3),
                         include_top=False,
                         weights='imagenet')
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

# Décongeler les 50 dernières couches du modèle de base
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Ajouter une couche Global Average Pooling
global_average_layer = layers.GlobalAveragePooling2D()(base_model.output)

# Ajouter des couches fully connected supplémentaires avec Dropout
dense_1 = layers.Dense(1024, activation='relu')(global_average_layer)
dropout_1 = layers.Dropout(0.5)(dense_1)

dense_2 = layers.Dense(512, activation='relu')(dropout_1)
dropout_2 = layers.Dropout(0.5)(dense_2)

dense_3 = layers.Dense(256, activation='relu')(dropout_2)
dropout_3 = layers.Dropout(0.5)(dense_3)

dense_4 = layers.Dense(128, activation='relu')(dropout_3)
dropout_4 = layers.Dropout(0.5)(dense_4)

# Créer les sorties pour chaque nutriment (13 au total)
outputs = []
for nutrient in range(13):
    output = layers.Dense(3, activation='softmax', name=f'nutrient_{nutrient}')(dropout_4)
    outputs.append(output)

# Créer le modèle final avec DenseNet201 en entrée et les 13 sorties en sortie
model = models.Model(inputs=base_model.input, outputs=outputs)

# Compiler le modèle avec des métriques adaptées pour chaque sortie
metrics = ['accuracy', Precision(name='precision'), Recall(name='recall')]
metrics_list = [metrics] * 13  # Appliquer les métriques à chaque nutriment

model.compile(optimizer='adam',
              loss=['categorical_crossentropy'] * 13,  # Chaque sortie utilise l'entropie croisée
              metrics=metrics_list)

# Afficher un résumé du modèle pour vérifier les couches et les sorties
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ zero_padding2d                │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,408 │ zero_padding2d[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ zero_padding2d_1              │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1 (MaxPooling2D)          │ (None, 56, 56, 64)        │               0 │ zero_padding2d_1[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_bn             │ (None, 56, 56, 64)        │             256 │ pool1[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_0_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 128)       │           8,192 │ conv2_block1_0_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 128)       │             512 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 128)       │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 32)        │          36,864 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_concat           │ (None, 56, 56, 96)        │               0 │ pool1[0][0],               │
│ (Concatenate)                 │                           │               

 Total params: 20,983,143 (80.04 MB)

 Trainable params: 4,552,167 (17.37 MB)

 Non-trainable params: 16,430,976 (62.68 MB)

In [6]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir le checkpoint pour sauvegarder les meilleurs poids
checkpoint = ModelCheckpoint('DenseNet201_best_weights.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             mode='max',
                             save_best_only=True)

# Early stopping pour arrêter l'entraînement si la validation stagne
early = EarlyStopping(monitor="val_loss",
                      mode="min",
                      restore_best_weights=True,
                      patience=5)

# Liste des callbacks
callbacks_list = [checkpoint, early]

In [7]:
import time
# Entraînement du modèle tout en mesurant le temps
start_time = time.time()

history = model.fit(
    train_val_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks_list,
    verbose=True,
    shuffle=True
)

end_time = time.time()

# Afficher le temps d'entraînement
training_time = end_time - start_time
print(f"Temps d'apprentissage : {training_time} secondes")

# Calcul manuel du F1-score après l'entraînement
precision = history.history['precision'][-1]
recall = history.history['recall'][-1]
f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon()) 
print(f'F1 Score: {f1:.4f}')

Epoch 1/15


C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\optimizers\base_optimizer.py:678: UserWarning: Gradients do not exist for variables ['kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


289/289 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - loss: 1.2243 - nutrient_0_accuracy: 0.4731 - nutrient_0_precision: 0.5025 - nutrient_0_recall: 0.3798

C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\callbacks\model_checkpoint.py:206: UserWarning: Can save best model only with val_accuracy available, skipping.
  self._save_model(epoch=epoch, batch=None, logs=logs)


289/289 ━━━━━━━━━━━━━━━━━━━━ 1858s 6s/step - loss: 1.2233 - nutrient_0_accuracy: 0.4734 - nutrient_0_precision: 0.5028 - nutrient_0_recall: 0.3800 - val_loss: 0.7212 - val_nutrient_0_accuracy: 0.6450 - val_nutrient_0_precision: 0.6951 - val_nutrient_0_recall: 0.5382
Epoch 2/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1771s 6s/step - loss: 0.6387 - nutrient_0_accuracy: 0.6496 - nutrient_0_precision: 0.6751 - nutrient_0_recall: 0.5898 - val_loss: 0.5804 - val_nutrient_0_accuracy: 0.6962 - val_nutrient_0_precision: 0.7087 - val_nutrient_0_recall: 0.6146
Epoch 3/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1791s 6s/step - loss: 0.5544 - nutrient_0_accuracy: 0.6780 - nutrient_0_precision: 0.6924 - nutrient_0_recall: 0.6377 - val_loss: 0.7017 - val_nutrient_0_accuracy: 0.6424 - val_nutrient_0_precision: 0.6726 - val_nutrient_0_recall: 0.5885
Epoch 4/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 3191s 11s/step - loss: 0.5221 - nutrient_0_accuracy: 0.7042 - nutrient_0_precision: 0.7130 - nutrient_0_recall: 0.6723 - val_loss: 0.4842

KeyError: 'precision'

In [8]:
model.save("model/DenseNet201_02_11.h5")

In [9]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 399s 11s/step - loss: 0.3930 - nutrient_0_accuracy: 0.7878 - nutrient_0_precision: 0.7895 - nutrient_0_recall: 0.7826
Résultats de l'évaluation : [0.40683531761169434, 0.7802768349647522, 0.7818499207496643, 0.775086522102356]


In [10]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

37/37 ━━━━━━━━━━━━━━━━━━━━ 420s 11s/step - loss: 0.3866 - nutrient_0_accuracy: 0.7917 - nutrient_0_precision: 0.7913 - nutrient_0_recall: 0.7892
Nombre total de résultats: 4
Résultats de l'évaluation: [0.40077778697013855, 0.7846021056175232, 0.7850304841995239, 0.7802768349647522]
La structure des résultats est différente de celle attendue.


In [12]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_mixte)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

60/60 ━━━━━━━━━━━━━━━━━━━━ 853s 14s/step - loss: 2.8311 - nutrient_0_accuracy: 0.4668 - nutrient_0_precision: 0.4706 - nutrient_0_recall: 0.4457
Résultats de l'évaluation : [2.767561674118042, 0.4602510333061218, 0.46515485644340515, 0.4398535490036011]


In [13]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_mixte)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

60/60 ━━━━━━━━━━━━━━━━━━━━ 848s 14s/step - loss: 2.7561 - nutrient_0_accuracy: 0.4528 - nutrient_0_precision: 0.4559 - nutrient_0_recall: 0.4301
Nombre total de résultats: 4
Résultats de l'évaluation: [2.772669792175293, 0.4576359689235687, 0.4620077610015869, 0.4356694519519806]
La structure des résultats est différente de celle attendue.


In [14]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_maiis)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

36/36 ━━━━━━━━━━━━━━━━━━━━ 512s 14s/step - loss: 2.3202 - nutrient_0_accuracy: 0.4704 - nutrient_0_precision: 0.4753 - nutrient_0_recall: 0.4479
Résultats de l'évaluation : [2.5669238567352295, 0.4571678340435028, 0.4606116712093353, 0.434440553188324]


In [15]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_maiis)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 504s 14s/step - loss: 2.1055 - nutrient_0_accuracy: 0.4633 - nutrient_0_precision: 0.4608 - nutrient_0_recall: 0.4380
Nombre total de résultats: 4
Résultats de l'évaluation: [2.6055281162261963, 0.4545454680919647, 0.4551341235637665, 0.4300699234008789]
La structure des résultats est différente de celle attendue.


In [18]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 509s 14s/step - loss: 0.4079 - nutrient_0_accuracy: 0.7796 - nutrient_0_precision: 0.7827 - nutrient_0_recall: 0.7773
Nombre total de résultats: 4
Résultats de l'évaluation: [0.41021403670310974, 0.7838541865348816, 0.7901667952537537, 0.78125]
La structure des résultats est différente de celle attendue.


In [19]:
import time
from tensorflow.keras import backend as K

# Fonction pour calculer la moyenne d'une métrique
def mean_metric(metric_values):
    return sum(metric_values) / len(metric_values)

# Nombre de nutriments
num_nutrients = 13

# Fonction pour calculer les moyennes pour chaque nutriment
def mean_nutrient_metric(metric_name, history):
    metrics = []
    for nutrient in range(num_nutrients):
        key = f'nutrient_{nutrient}_{metric_name}'
        if key in history:
            metrics.append(history[key])
    return [mean_metric(metric) for metric in zip(*metrics)]  # Moyenne sur les époques

# Calcul des moyennes pour l'ensemble des époques (entraînement)
mean_train_loss = mean_metric(history.history['loss'])
mean_train_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_train_precision = mean_nutrient_metric('precision', history.history)
mean_train_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (entraînement)
f1_train_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                   for p, r in zip(mean_train_precision, mean_train_recall)]
mean_f1_train = mean_metric(f1_train_scores)

# Calcul des moyennes pour l'ensemble des époques (validation)
mean_val_loss = mean_metric(history.history['val_loss'])
mean_val_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_val_precision = mean_nutrient_metric('precision', history.history)
mean_val_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (validation)
f1_val_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                 for p, r in zip(mean_val_precision, mean_val_recall)]
mean_f1_val = mean_metric(f1_val_scores)

# Affichage des résultats
print(f"Moyenne de la perte sur l'ensemble d'entraînement : {mean_train_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble d'entraînement : {mean_metric(mean_train_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble d'entraînement : {mean_metric(mean_train_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble d'entraînement : {mean_metric(mean_train_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble d'entraînement : {mean_f1_train:.4f}")

print(f"Moyenne de la perte sur l'ensemble de validation : {mean_val_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble de validation : {mean_metric(mean_val_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble de validation : {mean_metric(mean_val_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble de validation : {mean_metric(mean_val_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble de validation : {mean_f1_val:.4f}")

Moyenne de la perte sur l'ensemble d'entraînement : 0.4990
Moyenne de l'accuracy sur l'ensemble d'entraînement : 0.7326
Moyenne de la précision sur l'ensemble d'entraînement : 0.7408
Moyenne du rappel sur l'ensemble d'entraînement : 0.7103
Moyenne du F1-score sur l'ensemble d'entraînement : 0.7246
Moyenne de la perte sur l'ensemble de validation : 0.4935
Moyenne de l'accuracy sur l'ensemble de validation : 0.7326
Moyenne de la précision sur l'ensemble de validation : 0.7408
Moyenne du rappel sur l'ensemble de validation : 0.7103
Moyenne du F1-score sur l'ensemble de validation : 0.7246
